# Notebook 04 — DistilBETO con augmentación textual


## Por qué hago este experimento

En el notebook 03 entrené `distilbert-base-uncased`, que es un modelo en **inglés**, sobre mi corpus de informes mamográficos en **español**. Eso introdujo un sesgo metodológico que un revisor podría cuestionar: ¿el bajo Macro F1 del Transformer (0,5738) fue por una limitación intrínseca del modelo profundo en este corpus, o simplemente porque le pedí entender un idioma para el que no fue entrenado?

Para responderlo, replico el mismo experimento cambiando **solo** el modelo base a `dccuchile/distilbert-base-spanish-uncased` (DistilBETO de la Universidad de Chile).

---

## Paso 1 — Imports y configuración del entorno

**Qué espero ver**: `Device: MPS (Apple Silicon)`.

In [2]:
import time
inicio_global = time.time()

import os
import numpy as np
import pandas as pd
import torch
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Device: MPS (Apple Silicon)")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("Device: CUDA")
else:
    device = torch.device("cpu")
    print("Device: CPU (entrenamiento sera lento)")

MODEL_NAME = "dccuchile/distilbert-base-spanish-uncased"
DATA_PATH = "../data/processed/reports_cleaned.csv"
OUTPUT_DIR = "./results_distilbeto"
MAX_LENGTH = 256
BATCH_SIZE = 8
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3

print(f"\nModelo base: {MODEL_NAME}")
print(f"Hiperparametros: max_length={MAX_LENGTH}, batch={BATCH_SIZE}, lr={LEARNING_RATE}, epochs={NUM_EPOCHS}")

Device: MPS (Apple Silicon)

Modelo base: dccuchile/distilbert-base-spanish-uncased
Hiperparametros: max_length=256, batch=8, lr=2e-05, epochs=3


---

## Paso 2 — Cargo el corpus limpio

In [5]:
df = pd.read_csv(DATA_PATH)
print(f"Filas: {len(df)}")
print(f"Columnas: {df.columns.tolist()}")

assert "Full_Report_clean" in df.columns, "Falta Full_Report_clean"
assert "BI-RADS" in df.columns, "Falta BI-RADS"

print("\nDistribucion BI-RADS:")
print(df["BI-RADS"].value_counts().sort_index())

Filas: 4357
Columnas: ['Full_Report', 'Conclusion', 'Recommendations', 'BI-RADS', 'Full_Report_clean', 'Conclusion_clean', 'Recommendations_clean', 'len_report']

Distribucion BI-RADS:
BI-RADS
0     966
1     596
2    2635
3      87
4      52
5      16
6       5
Name: count, dtype: int64


---

## Paso 3 — Split estratificado 80/20

Mismo `random_state=42` que nb 02 y 03 para comparación directa.

In [6]:
X = df["Full_Report_clean"].astype(str).values
y = df["BI-RADS"].astype(int).values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")
print(f"Clases en train: {sorted(np.unique(y_train))}")
print(f"\nDistribucion train:")
for cls, count in sorted(Counter(y_train).items()):
    print(f"  BI-RADS {cls}: {count} ({100*count/len(y_train):.1f}%)")

Train: 3485 | Test: 872
Clases en train: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]

Distribucion train:
  BI-RADS 0: 773 (22.2%)
  BI-RADS 1: 477 (13.7%)
  BI-RADS 2: 2108 (60.5%)
  BI-RADS 3: 69 (2.0%)
  BI-RADS 4: 41 (1.2%)
  BI-RADS 5: 13 (0.4%)
  BI-RADS 6: 4 (0.1%)


---

## Paso 4 — Augmentación textual

Misma estrategia que nb 03: clases con <300 ejemplos se augmentan reemplazando palabras por sinónimos clínicos.

In [7]:
import re
import random
random.seed(SEED)

SINONIMOS = {
    "normal": ["habitual", "sin alteraciones", "conservado"],
    "aumento": ["incremento", "elevación", "mayor volumen"],
    "disminución": ["reducción", "descenso", "menor"],
    "presencia": ["existencia", "hallazgo", "detección"],
    "ausencia": ["no evidencia", "no se observa", "carencia"],
    "lesión": ["hallazgo", "alteración", "lesión"],
    "masa": ["formación", "lesión nodular", "masa"],
    "bordes": ["contornos", "límites", "márgenes"],
    "compatible": ["sugestivo", "concordante", "compatible"],
    "sospechoso": ["sugestivo", "de alerta", "sospechoso"]
}

def augmentar_texto(texto, n_variantes=1):
    variantes = []
    palabras = texto.split()
    for _ in range(n_variantes):
        nuevas = []
        for w in palabras:
            if w in SINONIMOS and random.random() < 0.4:
                nuevas.append(random.choice(SINONIMOS[w]))
            else:
                nuevas.append(w)
        variantes.append(" ".join(nuevas))
    return variantes

N_OBJETIVO = 300
clases_count = Counter(y_train)

print("Antes:")
for cls, count in sorted(clases_count.items()):
    print(f"  BI-RADS {cls}: {count}")

X_aug, y_aug = list(X_train), list(y_train)
for clase, count in clases_count.items():
    if count < N_OBJETIVO:
        faltan = N_OBJETIVO - count
        idx_clase = np.where(y_train == clase)[0]
        variantes_por_ejemplo = max(1, faltan // len(idx_clase) + 1)
        for idx in idx_clase:
            nuevas = augmentar_texto(X_train[idx], n_variantes=variantes_por_ejemplo)
            X_aug.extend(nuevas)
            y_aug.extend([clase] * len(nuevas))

X_aug = np.array(X_aug)
y_aug = np.array(y_aug)

print(f"\nDespues:")
for cls, count in sorted(Counter(y_aug).items()):
    print(f"  BI-RADS {cls}: {count}")
print(f"\nTotal: {len(X_train)} -> {len(X_aug)}")

Antes:
  BI-RADS 0: 773
  BI-RADS 1: 477
  BI-RADS 2: 2108
  BI-RADS 3: 69
  BI-RADS 4: 41
  BI-RADS 5: 13
  BI-RADS 6: 4

Despues:
  BI-RADS 0: 773
  BI-RADS 1: 477
  BI-RADS 2: 2108
  BI-RADS 3: 345
  BI-RADS 4: 328
  BI-RADS 5: 312
  BI-RADS 6: 304

Total: 3485 -> 4647


---

## ✋ Checkpoint 1 — Validación pasiva (asserts, no entrena)

In [8]:
assert 4000 <= len(X_aug) <= 30000, f"Tamano raro: {len(X_aug)}"
assert min(Counter(y_aug).values()) >= 50, "Alguna clase quedo muy pequena"
print("Checkpoint 1 OK")

Checkpoint 1 OK


---

## Paso 5 — Tokenización con DistilBETO

In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Vocab size: {tokenizer.vocab_size}")

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH, padding=False)

ds_train = Dataset.from_dict({"text": X_aug.tolist(), "label": y_aug.tolist()})
ds_test  = Dataset.from_dict({"text": X_test.tolist(), "label": y_test.tolist()})

ds_train = ds_train.map(tokenize_fn, batched=True)
ds_test  = ds_test.map(tokenize_fn, batched=True)

print(f"Primer ejemplo: {len(ds_train[0]['input_ids'])} tokens")

Vocab size: 31002


Map:   0%|          | 0/4647 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Primer ejemplo: 189 tokens


---

## Paso 6 — Inicialización del modelo

**Importante**: este modelo se inicializa **una sola vez** y NO se vuelve a tocar hasta el Trainer del Paso 8. Cualquier reasignación posterior es lo que rompió el experimento anterior.

In [10]:
NUM_LABELS = 7
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label={i: f"BI-RADS_{i}" for i in range(NUM_LABELS)},
    label2id={f"BI-RADS_{i}": i for i in range(NUM_LABELS)},
)
total_params = sum(p.numel() for p in model.parameters())
print(f"Parametros totales: {total_params:,}")
print("Modelo inicializado correctamente")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at dccuchile/distilbert-base-spanish-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Parametros totales: 67,327,495
Modelo inicializado correctamente


---

## Paso 7 — Función de métricas

In [11]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "weighted_f1": f1_score(labels, preds, average="weighted", zero_division=0),
    }
print("Metricas listas")

Metricas listas


---

## Paso 8 — Configuración del Trainer

Después de esta celda, el `trainer` está ligado al `model`. **No volveremos a reasignar `trainer.model` por ningún motivo**.

In [12]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=50,
    report_to="none",
    seed=SEED,
    fp16=False,
    dataloader_pin_memory=False,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_train,
    eval_dataset=ds_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
print("Trainer configurado")

/var/folders/3r/zv202tcx70l9blnhmnpbw1kc0000gn/T/ipykernel_6136/2987383968.py:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Trainer configurado


---

## ✋ Checkpoint 2 — Estimación pasiva de tiempo (NO entrena)

Esta vez la estimación es **puramente calculada**, sin ejecutar ningún paso del trainer. Uso el hecho conocido de que en MPS con DistilBETO y batch=8 cada paso toma ~150-300 ms.

> En tu corrida anterior, el entrenamiento completo tomó **4:41 minutos** para 1743 pasos. Es decir, ~161 ms/paso. Eso significa que en tu hardware el entrenamiento es **muy rápido**.

In [13]:
# Calculo pasivo - sin ejecutar nada del trainer
total_steps = (len(ds_train) // BATCH_SIZE) * NUM_EPOCHS
ms_por_paso_estimado = 200  # MPS con DistilBETO batch=8
estimacion_min = (total_steps * ms_por_paso_estimado) / 1000 / 60

print(f"Pasos totales estimados: {total_steps}")
print(f"Estimacion conservadora: ~{estimacion_min:.1f} min")
print(f"\n(En tu corrida anterior con bug fueron 4:41 min - este sera similar)")
print(f"\nProcedemos directo al Paso 9. NO reasignamos el modelo.")

Pasos totales estimados: 1740
Estimacion conservadora: ~5.8 min

(En tu corrida anterior con bug fueron 4:41 min - este sera similar)

Procedemos directo al Paso 9. NO reasignamos el modelo.


---

## Paso 9 — Entrenamiento completo (limpio)

Lanzo 3 épocas. **Validar época a época**:
- Training loss debe bajar progresivamente.
- **Validation loss debe bajar también** (si se queda fija en 1.89 como antes, hay otro problema).
- Macro F1 debe subir.

In [14]:
t0 = time.time()
trainer.train()
tiempo_entrenamiento = (time.time() - t0) / 60
print(f"\nEntrenamiento completado en {tiempo_entrenamiento:.1f} min")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 1}.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.077500,0.071257,0.980505,0.910689,0.981569
2,0.022300,0.082062,0.981651,0.938502,0.981871
3,0.008200,0.078908,0.981651,0.938625,0.982153



Entrenamiento completado en 4.7 min


**Validación crítica**: revisa la tabla de épocas que imprime Hugging Face. Si validation loss baja de ~1.9 a algo como ~0.3-0.6 y Macro F1 sube a >0.50, el experimento está bien. Si validation loss se queda constante, parar y avisarme.

---

## Paso 10 — Evaluación final sobre test

In [15]:
eval_result = trainer.evaluate(eval_dataset=ds_test)

print("=" * 60)
print("RESULTADOS FINALES - DistilBETO + augmentacion")
print("=" * 60)
for k, v in eval_result.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

predictions = trainer.predict(ds_test)
y_pred = np.argmax(predictions.predictions, axis=-1)

print("\nReporte por clase:")
print(classification_report(y_test, y_pred, zero_division=0, digits=4))

RESULTADOS FINALES - DistilBETO + augmentacion
  eval_loss: 0.0789
  eval_accuracy: 0.9817
  eval_macro_f1: 0.9386
  eval_weighted_f1: 0.9822
  eval_runtime: 4.6633
  eval_samples_per_second: 186.9900
  eval_steps_per_second: 23.3740
  epoch: 3.0000

Reporte por clase:
              precision    recall  f1-score   support

           0     0.9895    0.9793    0.9844       193
           1     1.0000    0.9916    0.9958       119
           2     0.9886    0.9867    0.9877       527
           3     0.8333    0.8333    0.8333        18
           4     0.6667    0.9091    0.7692        11
           5     1.0000    1.0000    1.0000         3
           6     1.0000    1.0000    1.0000         1

    accuracy                         0.9817       872
   macro avg     0.9254    0.9571    0.9386       872
weighted avg     0.9831    0.9817    0.9822       872



---

## Paso 11 — Comparación contra el ganador actual

In [16]:
macro_f1_distilbeto = eval_result["eval_macro_f1"]
macro_f1_linsvc = 0.9367
macro_f1_distilbert_eng = 0.5738

print("=" * 60)
print("COMPARACION MULTI-MODELO (Macro F1)")
print("=" * 60)
print(f"  LinearSVC balanced (clasico):           {macro_f1_linsvc:.4f}")
print(f"  DistilBETO + aug (este nb, espanol):    {macro_f1_distilbeto:.4f}")
print(f"  DistilBERT + aug (nb 03, ingles):       {macro_f1_distilbert_eng:.4f}")

delta_vs_linsvc = macro_f1_distilbeto - macro_f1_linsvc
delta_vs_eng = macro_f1_distilbeto - macro_f1_distilbert_eng

print(f"\n  DistilBETO vs LinearSVC:    {delta_vs_linsvc:+.4f}")
print(f"  DistilBETO vs DistilBERT-en: {delta_vs_eng:+.4f}")

COMPARACION MULTI-MODELO (Macro F1)
  LinearSVC balanced (clasico):           0.9367
  DistilBETO + aug (este nb, espanol):    0.9386
  DistilBERT + aug (nb 03, ingles):       0.5738

  DistilBETO vs LinearSVC:    +0.0019
  DistilBETO vs DistilBERT-en: +0.3648


---

## Paso 12 — Guardar resultados

In [17]:
import json
from pathlib import Path

Path(OUTPUT_DIR).mkdir(exist_ok=True)

resumen = {
    "modelo": MODEL_NAME,
    "estrategia": "fine-tuning + augmentacion textual",
    "hiperparametros": {
        "max_length": MAX_LENGTH,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "num_epochs": NUM_EPOCHS,
    },
    "tamano_train_augmentado": int(len(X_aug)),
    "tamano_test": int(len(X_test)),
    "metricas": {k: float(v) for k, v in eval_result.items() if isinstance(v, (int, float))},
    "tiempo_entrenamiento_min": round(tiempo_entrenamiento, 2),
    "tiempo_total_min": round((time.time() - inicio_global) / 60, 2),
}

with open(f"{OUTPUT_DIR}/resumen_distilbeto.json", "w", encoding="utf-8") as f:
    json.dump(resumen, f, indent=2, ensure_ascii=False)

print(f"Guardado en {OUTPUT_DIR}/resumen_distilbeto.json")
print(f"Tiempo total notebook: {resumen['tiempo_total_min']:.1f} min")

print("\nFILA PARA LA TABLA 5 DEL INFORME:")
print(f"  Modelo:      DistilBETO + augmentacion (espanol)")
print(f"  Accuracy:    {eval_result['eval_accuracy']:.4f}")
print(f"  Macro F1:    {eval_result['eval_macro_f1']:.4f}")
print(f"  Weighted F1: {eval_result['eval_weighted_f1']:.4f}")

Guardado en ./results_distilbeto/resumen_distilbeto.json
Tiempo total notebook: 15.4 min

FILA PARA LA TABLA 5 DEL INFORME:
  Modelo:      DistilBETO + augmentacion (espanol)
  Accuracy:    0.9817
  Macro F1:    0.9386
  Weighted F1: 0.9822


## Conclusiones del experimento

### Resultados obtenidos

Tras entrenar DistilBETO (`dccuchile/distilbert-base-spanish-uncased`) con la misma estrategia de augmentación textual que apliqué en el notebook 03 al DistilBERT en inglés, obtengo los siguientes resultados sobre el conjunto de prueba (872 informes):

- **Accuracy**: 0,9817
- **Macro F1**: 0,9386
- **Weighted F1**: 0,9822
- **Eval Loss**: 0,0789
- **Tiempo de entrenamiento**: ~5 min en MPS (Apple Silicon)

### Validación de las hipótesis planteadas al inicio

Al comparar este resultado con mis dos modelos de referencia, puedo evaluar las tres hipótesis que formulé al inicio del notebook:

| Modelo | Macro F1 | Diferencia |
|---|---|---|
| DistilBERT inglés + augmentación (nb 03) | 0,5738 | baseline |
| LinearSVC balanced (nb 02, ganador previo) | 0,9367 | +0,3629 |
| **DistilBETO + augmentación (nb 04)** | **0,9386** | **+0,3648** |

- **H1 (DistilBETO mejora claramente respecto a su contraparte inglesa pero queda por debajo de LinearSVC)** → parcialmente confirmada: efectivamente mejora +0,3648 respecto a la versión inglesa, pero **no queda por debajo de LinearSVC**: lo iguala e incluso lo supera marginalmente.
- **H2 (DistilBETO supera a LinearSVC)** → confirmada por márgen estrecho a nivel global (+0,0019), pero el desempeño por clase muestra una ventaja más clara en las categorías clínicamente críticas.
- **H3 (no se observa mejora respecto al inglés)** → rechazada: el cambio de idioma del modelo base sí tuvo un impacto sustantivo.

### Lectura por clase BI-RADS

El reporte por clase es donde el resultado se vuelve clínicamente significativo:

| BI-RADS | Categoría clínica | F1 DistilBETO | Soporte (test) |
|---|---|---|---|
| 0 | Incompleto | 0,9844 | 193 |
| 1 | Negativo | 0,9958 | 119 |
| 2 | Benigno | 0,9877 | 527 |
| 3 | Probablemente benigno | 0,8333 | 18 |
| 4 | Sospecha de malignidad | 0,7692 | 11 |
| 5 | Altamente sospechoso | 1,0000 | 3 |
| 6 | Malignidad confirmada | 1,0000 | 1 |

El hallazgo más relevante es que DistilBETO resuelve razonablemente las categorías 3 y 4, que son las más sensibles en términos de seguimiento clínico:

- **BI-RADS 3 (F1 = 0,8333)**: estable con 18 muestras de test, es un valor robusto.
- **BI-RADS 4 (F1 = 0,7692)**: con 11 muestras, sigue siendo informativo y representa el principal logro frente al modelo clásico, ya que en el modelo LinearSVC esta clase era el principal punto de confusión.

Los F1 = 1,0000 de las clases 5 y 6 deben interpretarse con cautela: con solo 3 y 1 muestra en el test respectivamente, esos valores no son estadísticamente significativos y podrían ser ruido. La conclusión defendible es que el modelo no falló en esas pocas muestras, pero no puedo afirmar que tenga buen desempeño general en ellas sin más datos.

### Decisión para el MVP

A la luz de estos resultados, decido **cambiar el modelo base del MVP de LinearSVC balanced a DistilBETO con augmentación**. La justificación es doble:

1. **Empate técnico en macro F1 global**, ligeramente a favor de DistilBETO (0,9386 vs. 0,9367).
2. **Ventaja en las clases clínicamente críticas**, particularmente BI-RADS 4 (sospecha de malignidad), donde el costo de un falso negativo es alto.

Esta decisión implica trade-offs operacionales que documento como riesgos en el informe parcial: el modelo serializado pasa de ~5 MB a ~250 MB, el tiempo de carga sube de milisegundos a 2–5 segundos, y la predicción por informe pasa de microsegundos a ~10–100 ms según hardware. Estos costos siguen siendo perfectamente asumibles para un MVP de validación técnica en un computador local.

### Limitaciones metodológicas reconocidas

Por honestidad metodológica, dejo explícitos los siguientes reparos sobre este experimento:

1. **Tamaño del test en clases minoritarias**: con solo 3 muestras en BI-RADS 5 y 1 en BI-RADS 6, los resultados en estas clases son indicativos pero no estadísticamente robustos. Sería deseable validar con un test set más balanceado o mediante validación cruzada estratificada también para el Transformer.
2. **Comparación sobre un único split**: aunque el modelo LinearSVC ya fue validado con CV 5-fold, DistilBETO no. El empate técnico podría revertirse en cualquier dirección bajo otra partición.
3. **Augmentación textual con diccionario limitado**: el diccionario de sinónimos clínicos usado (~10 entradas) es simple. Una augmentación más rica (por ejemplo, vía back-translation o paráfrasis con LLM) podría modificar los resultados.
4. **Sin búsqueda de hiperparámetros**: usé los mismos hiperparámetros del nb 03 (lr=2e-5, batch=8, 3 epochs) por comparabilidad, pero no exploré configuraciones específicas para DistilBETO. Es posible que el desempeño pudiera mejorar aún más.

### Implicación para el informe parcial

Voy a actualizar la tabla 5 del informe parcial agregando esta fila:

| Modelo | Accuracy | Macro F1 | Weighted F1 | Familia |
|---|---|---|---|---|
| **DistilBETO + augmentación (español)** | **0,9817** | **0,9386** | **0,9822** | **Transformer** |

Y reescribiré la sección de comparación final indicando que, contrariamente a lo planeado, el modelo seleccionado para el MVP pasa a ser DistilBETO con augmentación, con la justificación clínica ya descrita.